In [2]:
import os
import sys
import shutil
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
import pandas as pd
import numpy as np
source_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
sys.path.append(source_path)

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
from torch.utils.data import DataLoader
import winsound
import itertools
import random
from datetime import datetime

In [3]:
custom_pretrained='original' #original
kind = 'patches_224'  # Example kind, can be changed
selected_FE = 'resnet50' #'clip-vit-large-patch14-inter' #'clip-vit-large-patch14-un' 'BEiT-Large' 'BEiT-Large-inter'	
#'clip-vit-large-patch14-inter'# #'trocr-base-stage1'#'clip-vit-large-patch14'#'DeiT-Tiny' 
# #'clip-vit-large-patch14'  # Example feature extractor, can be changed
if custom_pretrained=='original':
    save_common=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\representation_extraction\\torch_model_trained_on_rep\\'
else:
    save_common=source_path+f'\\outputs\\online_deep_feature_extraction\\{selected_FE}\\{custom_pretrained}\\torch_model_trained_on_rep\\'

In [4]:
search_type = 'grid_search'
file_id = '08-12'
prev_log = os.path.join(save_common, f'{search_type}_results_{file_id}.csv')
prev_results = pd.read_csv(prev_log)


In [14]:
prev_results.columns

Index(['best_val_loss', 'best_val_acc', 'best_epoch', 'best_train_loss',
       'best_train_acc', 'last_epoch', 'last_val_loss', 'last_val_acc',
       'last_train_loss', 'last_train_acc', 'lr', 'dropout', 'n_neurons',
       'model_name', 'optimizer', 'scheduler', 'log_grad_norm', 'activation',
       'with_input_norm', 'id', 'Accuracy for individual patches',
       'Accuracy for majority_vote', 'Accuracy for weighted_vote',
       'Accuracy for most_probable', 'Accuracy for writer level prediction'],
      dtype='object')

In [5]:
prev_results['generalization']= prev_results['best_train_acc'] - prev_results['best_val_acc']

In [10]:
hyperparams_cols=['lr','dropout','n_neurons','with_input_norm','last_epoch']
selected_cols=['generalization','best_val_acc',
       'Accuracy for majority_vote', 'Accuracy for weighted_vote',
       'Accuracy for most_probable','best_val_loss']+hyperparams_cols
threshold=0.04
prev_results[(prev_results['generalization']<=threshold) & 
             (prev_results['generalization']>=-threshold)][selected_cols].sort_values('best_val_loss', ascending=True).head(20)
#0.799296 weighted , gen 0.029190
#1e-4,0.1,128
#1e-4,0.4,16


,generalization,best_val_acc,Accuracy for majority_vote,Accuracy for weighted_vote,Accuracy for most_probable,best_val_loss,lr,dropout,n_neurons,with_input_norm,last_epoch
53,0.038072,0.677553,0.693662,0.690141,0.693662,0.586207,0.00010,0.4,512,batch_norm,3
15,0.035866,0.669630,0.707746,0.711268,0.697183,0.587878,0.00010,0.4,128,batch_norm,3
1,-0.010652,0.676232,0.683099,0.676056,0.672535,0.593124,0.00100,0.1,32,NaN,5
70,0.032066,0.668134,0.700704,0.686620,0.690141,0.594443,0.00010,0.4,64,batch_norm,3
102,0.025093,0.666197,0.707746,0.697183,0.686620,0.595162,0.00010,0.1,16,NaN,12
14,0.032934,0.665581,0.683099,0.693662,0.686620,0.597174,0.00010,0.1,256,NaN,11
10,0.020027,0.664437,0.686620,0.683099,0.676056,0.597745,0.00010,0.4,128,NaN,9
45,0.011446,0.670070,0.700704,0.700704,0.686620,0.597897,0.00100,0.1,512,batch_norm,3
48,0.036554,0.663468,0.686620,0.693662,0.704225,0.598095,0.00001,0.1,16,batch_norm,13
47,0.016428,0.666285,0.676056,0.676056,0.676056,0.598506,0.00100,0.1,256,batch_norm,3
